<a href="https://colab.research.google.com/github/raj-aryan7/ProteinBERT/blob/main/notebooks/LSTM_with_pretraining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install tape-proteins
!pip install lmdb
!pip install biopython
!pip install torch torchvision
!pip install pandas numpy scikit-learn tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.9/68.9 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 64.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.4/299.4 kB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 96.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 11.1 MB/s eta 0:00:00


In [2]:
!wget https://s3.amazonaws.com/songlabdata/proteindata/data_pytorch/secondary_structure.tar.gz
!tar -xzf secondary_structure.tar.gz

!mkdir -p data
!mv secondary_structure data/

--2026-03-09 19:06:57--  https://s3.amazonaws.com/songlabdata/proteindata/data_pytorch/secondary_structure.tar.gz
Resolving s3.amazonaws.com (s3.amazonaws.com)... 16.15.199.167, 16.182.36.232, 3.5.12.114, ...
Connecting to s3.amazonaws.com (s3.amazonaws.com)|16.15.199.167|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 251794897 (240M) [application/x-tar]
Saving to: ‘secondary_structure.tar.gz’

secondary_structure 100%[===================>] 240.13M  33.6MB/s    in 7.2s    

2026-03-09 19:07:05 (33.4 MB/s) - ‘secondary_structure.tar.gz’ saved [251794897/251794897]



In [3]:
import torch
import torch.nn as nn
import torch.optim as optim

import numpy as np

from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence

from tape.datasets import SecondaryStructureDataset
from tape import TAPETokenizer

from tqdm import tqdm

In [4]:
train_dataset = SecondaryStructureDataset(
    data_path='./data',
    split='train'
)

valid_dataset = SecondaryStructureDataset(
    data_path='./data',
    split='valid'
)

print("Train size:", len(train_dataset))
print("Valid size:", len(valid_dataset))

Train size: 8678
Valid size: 2170


In [5]:
tokenizer = TAPETokenizer(vocab='iupac')

In [6]:
def collate_fn(batch):

    sequences = []
    labels = []

    for item in batch:
        seq, mask, target = item

        sequences.append(torch.tensor(seq))
        labels.append(torch.tensor(target))

    sequences = pad_sequence(sequences, batch_first=True)
    labels = pad_sequence(labels, batch_first=True, padding_value=-1)

    return sequences, labels

In [7]:
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=collate_fn
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=collate_fn
)

In [8]:
class ProteinLSTMPretrain(nn.Module):

    def __init__(self, vocab_size=30, embed_dim=128, hidden_dim=256):

        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)

        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.fc = nn.Linear(hidden_dim*2, vocab_size)

    def forward(self, x):

        x = self.embedding(x)

        x,_ = self.lstm(x)

        out = self.fc(x)

        return out

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

pretrain_model = ProteinLSTMPretrain().to(device)

criterion = nn.CrossEntropyLoss(ignore_index=-1)

optimizer = optim.Adam(pretrain_model.parameters(), lr=1e-3)

In [17]:
epochs = 5

for epoch in range(epochs):

    pretrain_model.train()

    total_loss = 0

    for inputs, labels in tqdm(train_loader):

        inputs = inputs.to(device)

        optimizer.zero_grad()

        outputs = pretrain_model(inputs)

        loss = criterion(
            outputs.view(-1,30),
            inputs.view(-1)
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print("Pretrain Epoch",epoch+1,"Loss:",total_loss/len(train_loader))

100%|██████████| 543/543 [00:25<00:00, 21.02it/s]


Pretrain Epoch 1 Loss: 5.265795479423125e-05


100%|██████████| 543/543 [00:23<00:00, 23.25it/s]


Pretrain Epoch 2 Loss: 5.279770178730432e-05


100%|██████████| 543/543 [00:25<00:00, 21.60it/s]


Pretrain Epoch 3 Loss: 5.282547240237479e-05


100%|██████████| 543/543 [00:25<00:00, 20.99it/s]


Pretrain Epoch 4 Loss: 5.3130250103388135e-05


100%|██████████| 543/543 [00:26<00:00, 20.77it/s]

Pretrain Epoch 5 Loss: 5.250167609853297e-05


In [18]:
torch.save(pretrain_model.state_dict(), "pretrained_lstm.pt")

In [19]:
class LSTMSecondaryStructure(nn.Module):

    def __init__(self, vocab_size=30, embed_dim=128, hidden_dim=256, num_classes=3):

        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)

        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.fc = nn.Linear(hidden_dim*2, num_classes)

    def forward(self, x):

        x = self.embedding(x)

        x,_ = self.lstm(x)

        out = self.fc(x)

        return out

In [20]:
model = LSTMSecondaryStructure().to(device)

pretrained_dict = torch.load("pretrained_lstm.pt")

model.embedding.weight.data = pretrain_model.embedding.weight.data
model.lstm.weight_ih_l0.data = pretrain_model.lstm.weight_ih_l0.data

In [21]:
criterion = nn.CrossEntropyLoss(ignore_index=-1)

optimizer = optim.Adam(model.parameters(), lr=1e-4)

In [22]:
epochs = 5

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for inputs, labels in tqdm(train_loader):

        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)

        loss = criterion(
            outputs.view(-1,3),
            labels.view(-1)
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {total_loss/len(train_loader):.4f}")

100%|██████████| 543/543 [00:24<00:00, 22.61it/s]


Epoch 1 Loss: 0.8380


100%|██████████| 543/543 [00:23<00:00, 22.63it/s]


Epoch 2 Loss: 0.7757


100%|██████████| 543/543 [00:23<00:00, 22.92it/s]


Epoch 3 Loss: 0.7532


100%|██████████| 543/543 [00:23<00:00, 22.65it/s]


Epoch 4 Loss: 0.7328


100%|██████████| 543/543 [00:23<00:00, 22.77it/s]

Epoch 5 Loss: 0.7089


In [23]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for inputs, labels in valid_loader:

        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)

        preds = torch.argmax(outputs, dim=-1)

        mask = labels != -1

        correct += (preds[mask] == labels[mask]).sum().item()
        total += mask.sum().item()

accuracy = correct / total

print("Validation Accuracy:", accuracy)

Validation Accuracy: 0.6966087360911855
